                                                                Т2

In [51]:
import numpy as np
import pandas as pd
from scipy import stats
from itertools import combinations

# Данные
data = pd.DataFrame({
    'Group': [1, 1, 2, 2, 2, 2, 2, 2, 
              3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
              4, 4, 4, 4, 
              5, 5],
    'IgA': [83, 85, 84, 85, 85, 86, 86, 87,
            86, 87, 87, 87, 88, 88, 88, 88, 88, 89, 90,
            89, 90, 90, 91,
            90, 92]
})
alpha = 0.05

ПУНКТ а) Влияет ли возраст на содержание Ig A регрессионным анализом

In [52]:
# 1. Построение матрицы Ψ
n = len(data)                     # n = 25
PSI = np.column_stack([np.ones(n), data['Group'].values])  # Ψ = [1, Age]
Y = data['IgA'].values

# 2. Оценка коэффициентов β
# F = ΨᵀΨ
F = PSI.T @ PSI

# F⁻¹
F_inv = np.linalg.inv(F)

# β̂ = F⁻¹ Ψᵀ Y
beta_hat = F_inv @ (PSI.T @ Y)

beta_0 = beta_hat[0]  # свободный член
beta_1 = beta_hat[1]  # коэффициент при возрасте

print(f"Уравнение регрессии: η = {beta_0:.3f} + {beta_1:.3f}×x")

Y_hat = PSI @ beta_hat           # предсказанные значения
e = Y - Y_hat                    # остатки
RSS = e @ e

# 3. Оценка дисперсии ошибки
p = 2
sigma_hat = np.sqrt(RSS / (n-p)) # σ̂

# 4. Проверка гипотезы H₀: β₁ = 0 (возраст не влияет)

delta = beta_1 / (sigma_hat * np.sqrt(F_inv[1, 1]))

# p-value
p_value = 2 * (1 - stats.t.cdf(abs(delta), df=(n-p)))

print(f"\nПроверка гипотезы H₀: β₁ = 0:")
print(f"   delta = {delta}")
print(f"   p_value = {p_value}")
print("\nВЫВОД:")
if p_value < alpha:
    print("Отвергаем H₀, возраст значим")
if p_value > alpha:
    print("Нет основания отвергать H₀, возраст не значим")

Уравнение регрессии: η = 81.896 + 1.940×x

Проверка гипотезы H₀: β₁ = 0:
   delta = 9.467380449973723
   p_value = 2.1251511661546374e-09

ВЫВОД:
Отвергаем H₀, возраст значим


ПУНКТ б) Попарно сравнить средние в каждой группе

In [53]:
# 1. Cоздание dummy-переменных
# Определяем уникальные группы
groups = sorted(data['Group'].unique())  # [1, 2, 3, 4, 5]
k = len(groups)                          # k = 5
n = len(data)                            # n = 25

# Создаем пустую матрицу Ψ размера n × k
Psi = np.zeros((n, k))

# Заполняем: для каждого наблюдения ставим 1 в столбец, соответствующий группе
for i, group in enumerate(data['Group']):
    Psi[i, group - 1] = 1

# Вектор отклика Y
Y = data['IgA'].values

# 2. Оценка β
F = Psi.T @ Psi                     # F = ΨᵀΨ
F_inv = np.linalg.inv(F)            # F⁻¹
beta_hat = F_inv @ (Psi.T @ Y)      # β̂ = F⁻¹ Ψᵀ Y

print("\n2. Оценки коэффициентов (β̂):")
for i, g in enumerate(groups, 1):
    print(f"   Группа {g}: β̂_{g} = {beta_hat[i-1]:.3f}")

# 3. Оценка дисперсии ошибки (стр. 57)
Y_hat = Psi @ beta_hat
e = Y - Y_hat
RSS = e @ e
sigma2_hat = RSS / (n - k)             # σ̂²
sigma_hat = np.sqrt(sigma2_hat)      # σ̂

# 4. Размеры групп
group_sizes = [len(data[data['Group'] == g]) for g in groups]

# 5. Попарное сравнение средних
pairs = list(combinations(groups, 2))
m = len(pairs)

results = []

print(f"\n{'Пара':<8} {'p-value':<12}")
print("-" * 20)

# H_0 - средние в двух группах равны

for i, j in pairs:
    idx_i = i-1
    idx_j = j-1
    
    diff = beta_hat[idx_i] - beta_hat[idx_j]

    se_component = np.sqrt(RSS * (F_inv[idx_i, idx_i] + F_inv[idx_j, idx_j]))
    
    delta = diff / se_component * np.sqrt((n-k))
    
    p_value = 2 * (1 - stats.t.cdf(abs(delta), df=(n-k)))

    results.append({
        'pair': f"{i}-{j}",
        'diff': diff,
        't_stat': delta,
        'p_value': p_value
    })
    
    print(f"{i}-{j:<5} {p_value:<12.6f}")

# 7. Коррекция на множественность метод Холма-Бонферрони
print(f"\nМетод Холма-Бонферрони:")

sorted_results = sorted(results, key=lambda x: x['p_value'])

print(f"{'Ранг k':<8} {'Пара':<8} {'p-value':<12} {'α/(m-k+1)':<15} {'Результат':<15}")
print("-" * 65)

holm_rejected_pairs = []
for k_idx, r in enumerate(sorted_results, 1):
    holm_alpha = alpha / (m - k_idx + 1)
    is_sig = r['p_value'] < holm_alpha
    result_str = "отвергаем H₀" if is_sig else "принимаем H₀"
    print(f"{k_idx:<8} {r['pair']:<8} {r['p_value']:<12.6f} {holm_alpha:<15.5f} {result_str:<15}")
    if is_sig:
        holm_rejected_pairs.append(r['pair'])


2. Оценки коэффициентов (β̂):
   Группа 1: β̂_1 = 84.000
   Группа 2: β̂_2 = 85.500
   Группа 3: β̂_3 = 87.818
   Группа 4: β̂_4 = 90.000
   Группа 5: β̂_5 = 91.000

Пара     p-value     
--------------------
1-2     0.103100    
1-3     0.000166    
1-4     0.000003    
1-5     0.000002    
2-3     0.000395    
2-4     0.000003    
2-5     0.000004    
3-4     0.002393    
3-5     0.001003    
4-5     0.295791    

Метод Холма-Бонферрони:
Ранг k   Пара     p-value      α/(m-k+1)       Результат      
-----------------------------------------------------------------
1        1-5      0.000002     0.00500         отвергаем H₀   
2        2-4      0.000003     0.00556         отвергаем H₀   
3        1-4      0.000003     0.00625         отвергаем H₀   
4        2-5      0.000004     0.00714         отвергаем H₀   
5        1-3      0.000166     0.00833         отвергаем H₀   
6        2-3      0.000395     0.01000         отвергаем H₀   
7        3-5      0.001003     0.01250         о

In [54]:
# # ============================================================
# # 7. Доверительный интервал для β₁
# t_crit = stats.t.ppf(1 - alpha/2, df=df_resid)  # t_{0.975}(22) ≈ 2.074
# CI_lower = beta_1 - t_crit * SE_beta1
# CI_upper = beta_1 + t_crit * SE_beta1

# print(f"\n7. Доверительный интервал для β₁ (95%):")
# print(f"   β̂₁ ± t_{1-alpha/2}(n-p) × SE(β̂₁)")
# print(f"   {beta_1:.3f} ± {t_crit:.3f} × {SE_beta1:.4f}")
# print(f"   ДИ = ({CI_lower:.3f}, {CI_upper:.3f})")

# # ============================================================
# # 8. Коэффициент детерминации R² (стр. 63)
# # ============================================================
# TSS = np.sum((Y - np.mean(Y))**2)  # Total Sum of Squares
# R_squared = 1 - RSS / TSS

# print(f"\n8. Коэффициент детерминации (стр. 63):")
# print(f"   TSS = Σ(yᵢ - ȳ)² = {TSS:.4f}")
# print(f"   R² = 1 - RSS/TSS = 1 - {RSS:.4f}/{TSS:.4f} = {R_squared:.4f}")
# print(f"   → {R_squared*100:.1f}% вариации IgA объясняется возрастом")

# # ============================================================
# # 9. F-тест для всей регрессии (стр. 63)
# # ============================================================
# F_stat = ((TSS - RSS) / (p-1)) / (RSS / (n-p))
# F_pvalue = 1 - stats.f.cdf(F_stat, p-1, n-p)

# print(f"\n9. F-тест значимости всей регрессии (стр. 63):")
# print(f"   F = ((TSS-RSS)/(p-1)) / (RSS/(n-p)) = {F_stat:.3f}")
# print(f"   p-value = {F_pvalue:.6f}")

# # ============================================================
# # 10. ВЫВОД
# # ============================================================
# print("\n" + "=" * 60)
# print("10. ВЫВОД ПО ПУНКТУ (а)")
# print("=" * 60)

# if p_value < 0.05:
#     print("\n✅ Результат: ОТВЕРГАЕМ H₀ (β₁ ≠ 0)")
#     print(f"   p-value = {p_value:.6f} < 0.05")
#     print(f"\n   Возраст статистически значимо влияет на уровень IgA.")
#     print(f"   С каждой следующей возрастной группой IgA в среднем")
#     print(f"   увеличивается на {beta_1:.3f}% (95% ДИ: {CI_lower:.3f}–{CI_upper:.3f}).")
# else:
#     print("\n❌ Результат: НЕТ ОСНОВАНИЙ ОТВЕРГНУТЬ H₀")
#     print(f"   p-value = {p_value:.6f} ≥ 0.05")
#     print(f"\n   Влияние возраста на IgA статистически не значимо.")